# Combine the 6 use cases into one processed, model-ready parquet

**ELI5:** `explore_use_cases.ipynb` and `explore_usecase_definitions.ipynb` looked at
the 6 JSONL exports and their 6 `usecase.json` search briefs separately, and found a
punch list of things to fix before this data is model-ready (an `id` that collides
once use cases are pooled, inconsistent `year` dtypes, `venue` casing duplicates, a
`sources` field that's really several boolean facts squashed into one string, and
more). This notebook is where that punch list actually gets applied: read all 12 raw
files, fix every item, broadcast both files' metadata onto every paper row, and write
one combined `data/processed/papers_combined.parquet` — the file
`scripts/compare_embeddings.py` and `scripts/train_baseline_classifier.py` are meant
to consume via `--data`.

Rows are individual papers. Every column is either paper data or metadata broadcast
onto that paper's row (from the JSONL's own `_meta` line, or from its matching
`usecase.json`). This notebook shows its inputs, raises loudly on anything
unexpected rather than silently guessing, and prints the output metrics needed to
trust the result.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../../scripts")
from embedding_utils import POSITIVE_VALUES, NEGATIVE_VALUES  # noqa: E402 — see below

DATA_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
OUTPUT_PATH = PROCESSED_DIR / "papers_combined.parquet"

# Reusing embedding_utils' own constants rather than re-hardcoding "positive"/"negative"
# — the whole reason that module exists is so this kind of thing can't drift between
# the two analysis scripts and, now, this notebook.
TRIAGE_CATEGORIES = [*POSITIVE_VALUES, *NEGATIVE_VALUES, "pass"]
REVIEW_CATEGORIES = ["soft_pos", "soft_neg", "hard_pos", "hard_neg"]

## 1. Load (input)

Same loader pattern as the other two notebooks — split the JSONL's `_meta` line from
its paper records by content, not position — plus a small loader for the
`usecase.json` files. Matching between the two file types is by **name**
(`usecase.json`'s `name` vs. the export's `_meta.use_case`), confirmed exactly the way
`explore_usecase_definitions.ipynb` already validated it — raise immediately if that
match ever breaks, since everything below depends on it.

In [2]:
def load_jsonl_export(path: Path) -> tuple[dict, pd.DataFrame]:
    meta, rows = {}, []
    with open(path) as fh:
        for line in fh:
            d = json.loads(line)
            if set(d.keys()) == {"_meta"}:
                meta = d["_meta"]
            else:
                rows.append(d)
    return meta, pd.DataFrame(rows)


def load_usecase_def(path: Path) -> dict:
    return json.loads(path.read_text())


usecase_files = sorted(DATA_DIR.glob("*.usecase.json"))
jsonl_files = sorted(DATA_DIR.glob("*.jsonl"))

usecase_defs = {}
for f in usecase_files:
    d = load_usecase_def(f)
    usecase_defs[d["name"]] = {"file": f.name, "def": d}

jsonl_data = {}
for f in jsonl_files:
    meta, df = load_jsonl_export(f)
    jsonl_data[meta.get("use_case", f.stem)] = {"file": f.name, "meta": meta, "df": df}

print(f"Loaded {len(usecase_defs)} usecase.json files and {len(jsonl_data)} JSONL exports.")
for name, d in jsonl_data.items():
    print(f"  {name!r}: {len(d['df'])} papers ({d['file']})")

unmatched_usecase = set(usecase_defs) - set(jsonl_data)
unmatched_jsonl = set(jsonl_data) - set(usecase_defs)
if unmatched_usecase or unmatched_jsonl:
    raise ValueError(
        f"usecase.json/JSONL name mismatch — usecase.json only: {unmatched_usecase}, "
        f"JSONL only: {unmatched_jsonl}. Fix the mismatch before combining."
    )
print("Every usecase.json name matches exactly one JSONL export. OK to proceed.")

Loaded 6 usecase.json files and 6 JSONL exports.
  'Carbon Capture': 323 papers (carbon-capture-all.jsonl)
  'High quality microbial and fungal community in Soil': 602 papers (high-quality-microbial-and-fungal-community-in-soil-labelledFULLRUN.jsonl)
  'Low-carbon cement binders': 690 papers (low-carbon-cement-binders-all.jsonl)
  'Named Entity Recognition': 541 papers (named-entity-recognition-all.jsonl)
  'Predicting Technology Developments': 290 papers (predicting-technology-developments-all.jsonl)
  'solar cells for low earth orbit satellites': 427 papers (solar-cells-for-low-earth-orbit-satellites-all.jsonl)
Every usecase.json name matches exactly one JSONL export. OK to proceed.


## 2. Use case registry — key + consistent display name

A short, hand-curated code per use case (not an algorithmic slug — chosen for
shorter/nicer keys, at the cost of needing a manual entry here for any 7th use case
added later) plus a consistent display name. `name` is each use case's original
value unchanged, except solar-cells — the one use case whose name is inconsistently
all-lowercase everywhere else it appears — re-cased to Title Case here.

In [3]:
USE_CASE_REGISTRY = {
    "Carbon Capture": {
        "key": "carbon_capture", "name": "Carbon Capture"},
    "High quality microbial and fungal community in Soil": {
        "key": "soil_microbiome", "name": "High quality microbial and fungal community in Soil"},
    "Low-carbon cement binders": {
        "key": "cement_binders", "name": "Low-carbon cement binders"},
    "Named Entity Recognition": {
        "key": "ner", "name": "Named Entity Recognition"},
    "Predicting Technology Developments": {
        "key": "tech_forecasting", "name": "Predicting Technology Developments"},
    "solar cells for low earth orbit satellites": {
        "key": "solar_leo", "name": "Solar Cells for Low Earth Orbit Satellites"},
}

unregistered = set(jsonl_data) - set(USE_CASE_REGISTRY)
if unregistered:
    raise ValueError(
        f"No USE_CASE_REGISTRY entry for {unregistered} — add a key/name for this use "
        "case before combining (a new use case needs a deliberate short code, not a "
        "guessed one)."
    )

keys = [v["key"] for v in USE_CASE_REGISTRY.values()]
assert len(keys) == len(set(keys)), "USE_CASE_REGISTRY has a duplicate key — fix before proceeding."
print("Registry OK — every loaded use case has a key, and every key is unique:")
for name, reg in USE_CASE_REGISTRY.items():
    print(f"  {reg['key']:18s} <- {name!r}")

Registry OK — every loaded use case has a key, and every key is unique:
  carbon_capture     <- 'Carbon Capture'
  soil_microbiome    <- 'High quality microbial and fungal community in Soil'
  cement_binders     <- 'Low-carbon cement binders'
  ner                <- 'Named Entity Recognition'
  tech_forecasting   <- 'Predicting Technology Developments'
  solar_leo          <- 'solar cells for low earth orbit satellites'


## 3–5. Broadcasting both files' metadata onto every paper row

One function per concern, run once per use case:

- `flatten_usecase_def` — turns one `usecase.json`'s nested fields into the flat
  scalar/list values that get broadcast onto every row (list-valued fields like
  `terms.must_include` stay genuine Python lists — parquet stores these natively as
  `list<string>`, so no structure is lost).
- `build_use_case_frame` — does the actual per-paper work: renames `id` to
  `source_id` and adds the new global `paper_id` (`use_case_key__source_id`, fixing
  the cross-use-case `id` collisions `explore_use_cases.ipynb` found), broadcasts the
  JSONL's own `_meta` and the matching `usecase.json`'s fields, and applies every
  cleaning step from the punch list (dtypes, casing, categoricals, `sources`
  booleans, dropping `abstract_source`, the `has_abstract` flag, `language` nulls).

`source_tokens` (the individual values inside `sources`' comma-joined strings) is
computed once, up front, across all 6 files — so every use case's frame gets the
same set of `from_<source>` boolean columns, even a use case with zero papers from a
given source (that column is just all-`False` for it, which is what makes the 6
frames concatenable with identical schemas).

In [4]:
all_source_tokens = sorted({
    tok.strip()
    for d in jsonl_data.values()
    for combo in d["df"]["sources"].dropna().unique()
    for tok in combo.split(",")
    if tok.strip()
})
print("Source tokens found across all 6 exports:", all_source_tokens)


def flatten_usecase_def(d: dict) -> dict:
    """Flatten one usecase.json's fields into the values broadcast onto every row of
    its matching use case. List-valued fields stay as Python lists (native parquet
    list<string>), not joined strings — no structure lost."""
    trl = d["constraints"].get("trl")
    return {
        "problem_statement": d["problem_statement"],
        "objective": d["objective"],
        "usecase_schema_version": d["schema_version"],
        "domain_industry": d["domain"]["industry"],
        "domain_application": d["domain"]["application"],
        "domain_technology_focus": d["domain"]["technology_focus"],
        "terms_must_include": d["terms"]["must_include"],
        "terms_nice_to_have": d["terms"]["nice_to_have"],
        "terms_exclude": d["terms"]["exclude"],
        "performance_criteria": d["performance_criteria"],
        "trl_min": trl["min"] if trl else None,
        "trl_max": trl["max"] if trl else None,
        "constraints_scale": d["constraints"]["scale"],
        "constraints_cost": d["constraints"]["cost"],
        "decision_must_have": d["decision_criteria"]["must_have"],
        "decision_nice_to_have": d["decision_criteria"]["nice_to_have"],
        "decision_exclusions": d["decision_criteria"]["exclusions"],
        "decision_rules": d["decision_criteria"]["decision_rules"],
        "notes": d["notes"],
    }


def build_use_case_frame(name: str) -> pd.DataFrame:
    reg = USE_CASE_REGISTRY[name]
    df = jsonl_data[name]["df"].copy()
    meta = jsonl_data[name]["meta"]
    usecase_flat = flatten_usecase_def(usecase_defs[name]["def"])

    # --- identity: compound key, use-case key/name ---
    df = df.rename(columns={"id": "source_id"})
    df.insert(0, "paper_id", reg["key"] + "__" + df["source_id"].astype(str))
    df.insert(1, "use_case_key", reg["key"])
    df.insert(2, "use_case_name", reg["name"])

    # --- broadcast JSONL's own _meta (row 1) ---
    df["embed_model"] = meta.get("embed_model")
    df["embed_dim"] = meta.get("dim")
    df["exported_at"] = meta.get("exported_at")
    df["app_version"] = meta.get("app_version")

    # --- broadcast the matching usecase.json's fields ---
    for col, value in usecase_flat.items():
        df[col] = [value] * len(df) if isinstance(value, list) else value

    # --- dtypes: year / citation_count / trl_min / trl_max -> nullable Int64 ---
    # (broadcasting a bare scalar above doesn't enforce a dtype — trl_min/trl_max
    # would silently end up as `object` once concatenated with use cases that have
    # `None` here, unless cast explicitly, same as year/citation_count.)
    df["year"] = df["year"].astype("Int64")
    df["citation_count"] = df["citation_count"].astype("Int64")
    df["trl_min"] = df["trl_min"].astype("Int64")
    df["trl_max"] = df["trl_max"].astype("Int64")

    # --- venue casing ---
    df["venue"] = df["venue"].str.lower().str.strip()

    # --- language: nulls confirmed English by manual review ---
    df["language"] = df["language"].fillna("en")

    # --- sources: keep the raw joined string, add one boolean per source token ---
    source_sets = df["sources"].fillna("").apply(
        lambda s: {tok.strip() for tok in s.split(",") if tok.strip()}
    )
    for tok in all_source_tokens:
        df[f"from_{tok}"] = source_sets.apply(lambda s, tok=tok: tok in s)

    # --- triage_label / review_label: fixed Categoricals, null distinct from 'pass' ---
    unexpected_triage = set(df["triage_label"].dropna().unique()) - set(TRIAGE_CATEGORIES)
    if unexpected_triage:
        raise ValueError(f"{name}: unexpected triage_label value(s) {unexpected_triage}")
    df["triage_label"] = pd.Categorical(df["triage_label"], categories=TRIAGE_CATEGORIES)

    unexpected_review = set(df["review_label"].dropna().unique()) - set(REVIEW_CATEGORIES)
    if unexpected_review:
        raise ValueError(f"{name}: unexpected review_label value(s) {unexpected_review}")
    df["review_label"] = pd.Categorical(df["review_label"], categories=REVIEW_CATEGORIES)

    # --- abstract_source: always-null/inconsistently-present, drop entirely ---
    df = df.drop(columns=["abstract_source"], errors="ignore")

    # --- has_abstract flag; all rows kept, title-only rows just flagged ---
    df["has_abstract"] = df["abstract"].fillna("").str.split().str.len() > 0

    return df

Source tokens found across all 6 exports: ['arxiv', 'core', 'crossref', 'europe_pmc', 'openalex', 'pubmed', 'seed', 'semantic_scholar']


## 6. Run it — one frame per use case, then concatenate

Each use case is processed independently and the resulting frames' columns are
checked for an exact match before concatenating — this is where the `abstract_source`
schema gap (present in 5 exports, missing from the soil one) would have shown up as
an error, but it's moot now since that column is dropped in `build_use_case_frame`
before this check runs.

In [5]:
processed_frames = {}
for name in usecase_defs:
    frame = build_use_case_frame(name)
    processed_frames[name] = frame
    print(f"  {name!r}: {len(frame)} rows, {len(frame.columns)} columns")

reference_columns = None
for name, frame in processed_frames.items():
    if reference_columns is None:
        reference_columns = list(frame.columns)
    elif list(frame.columns) != reference_columns:
        raise ValueError(
            f"{name}'s processed columns don't match the others — "
            f"diff: {set(frame.columns) ^ set(reference_columns)}"
        )
print(f"\nAll {len(processed_frames)} processed frames share identical columns "
      f"({len(reference_columns)} columns). OK to concatenate.")

combined_df = pd.concat(processed_frames.values(), ignore_index=True)

  'Carbon Capture': 323 rows, 50 columns
  'High quality microbial and fungal community in Soil': 602 rows, 50 columns
  'Low-carbon cement binders': 690 rows, 50 columns
  'Named Entity Recognition': 541 rows, 50 columns
  'Predicting Technology Developments': 290 rows, 50 columns
  'solar cells for low earth orbit satellites': 427 rows, 50 columns

All 6 processed frames share identical columns (50 columns). OK to concatenate.


## 7. Validate + output metrics

Errors surface here as failed assertions, not silent bad data.

In [6]:
expected_rows = sum(len(f) for f in processed_frames.values())
assert len(combined_df) == expected_rows, (
    f"Row count mismatch: combined has {len(combined_df)}, expected {expected_rows}"
)
assert combined_df["paper_id"].is_unique, "paper_id is not globally unique — compound key is broken"
print(f"paper_id is globally unique across all {len(combined_df)} rows. "
      f"(id alone was NOT — {combined_df['source_id'].duplicated().sum()} collisions if "
      f"pooled without use_case_key.)")

print("\nPer-use-case row counts:")
for name, frame in processed_frames.items():
    print(f"  {USE_CASE_REGISTRY[name]['key']:18s} {len(frame):4d} rows")

print("\nFinal dtypes:")
print(combined_df.dtypes)

paper_id is globally unique across all 2873 rows. (id alone was NOT — 587 collisions if pooled without use_case_key.)

Per-use-case row counts:
  carbon_capture      323 rows
  soil_microbiome     602 rows
  cement_binders      690 rows
  ner                 541 rows
  tech_forecasting    290 rows
  solar_leo           427 rows

Final dtypes:
paper_id                        str
use_case_key                    str
use_case_name                   str
source_id                     int64
title                           str
authors                         str
year                          Int64
venue                           str
doi                             str
url                             str
sources                         str
language                        str
citation_count                Int64
triage_label               category
review_label               category
relevance_score             float64
abstract                        str
embedding                    object
embed_m

In [7]:
print("Null rate (%) per column, post-cleaning:")
null_rates = (combined_df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(null_rates[null_rates > 0])

print(f"\ncitation_count nulls (should be unchanged from the raw exports, "
      f"never filled with 0): {combined_df['citation_count'].isna().sum()}")
print(f"year nulls (should be unchanged from the raw exports; only the DTYPE was "
      f"fixed, not the null count): {combined_df['year'].isna().sum()}")
print(f"triage_label: never-triaged (null) vs. explicit 'pass' are distinct — "
      f"{combined_df['triage_label'].isna().sum()} null, "
      f"{(combined_df['triage_label'] == 'pass').sum()} 'pass'")

Null rate (%) per column, post-cleaning:
review_label      100.0
trl_min            42.3
trl_max            42.3
triage_label       16.6
citation_count     14.3
doi                 6.1
venue               1.7
authors             0.9
url                 0.5
year                0.5
dtype: float64

citation_count nulls (should be unchanged from the raw exports, never filled with 0): 410
year nulls (should be unchanged from the raw exports; only the DTYPE was fixed, not the null count): 15
triage_label: never-triaged (null) vs. explicit 'pass' are distinct — 478 null, 543 'pass'


## 8. Save

One deterministic filename — matching the existing scripts' convention of
overwriting a fixed output path so re-runs are diffable rather than piling up dated
files.

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
combined_df.to_parquet(OUTPUT_PATH, index=False)
size_kb = OUTPUT_PATH.stat().st_size / 1024
print(f"Saved {len(combined_df)} rows x {len(combined_df.columns)} columns to "
      f"{OUTPUT_PATH} ({size_kb:.1f} KB)")

Saved 2873 rows x 50 columns to ../../data/processed/papers_combined.parquet (9062.9 KB)


## Verify the round trip

Read the saved parquet back and re-check the things that matter — dtype fidelity
isn't guaranteed just because `to_parquet` didn't raise.

In [9]:
check_df = pd.read_parquet(OUTPUT_PATH)

assert len(check_df) == len(combined_df)
assert check_df["paper_id"].is_unique
assert str(check_df["year"].dtype) == "Int64"
assert str(check_df["citation_count"].dtype) == "Int64"
assert str(check_df["trl_min"].dtype) == "Int64"
assert str(check_df["trl_max"].dtype) == "Int64"
assert str(check_df["triage_label"].dtype) == "category"
assert str(check_df["review_label"].dtype) == "category"
assert "abstract_source" not in check_df.columns
assert check_df["use_case_key"].notna().all()
assert check_df["use_case_name"].notna().all()

print("Round-trip checks passed:")
print(f"  {len(check_df)} rows, {len(check_df.columns)} columns")
print(f"  triage_label categories: {list(check_df['triage_label'].cat.categories)}")
print(f"  review_label categories: {list(check_df['review_label'].cat.categories)}")
print(f"  use cases present: {sorted(check_df['use_case_key'].unique())}")

print("\nOne sample row per use case, to eyeball that broadcast metadata lines up "
      "with the right use case (no cross-wiring):")
sample = check_df.groupby("use_case_key", observed=True).first()[
    ["use_case_name", "domain_industry", "objective", "embed_model"]
]
sample

Round-trip checks passed:
  2873 rows, 50 columns
  triage_label categories: ['positive', 'negative', 'pass']
  review_label categories: ['soft_pos', 'soft_neg', 'hard_pos', 'hard_neg']
  use cases present: ['carbon_capture', 'cement_binders', 'ner', 'soil_microbiome', 'solar_leo', 'tech_forecasting']

One sample row per use case, to eyeball that broadcast metadata lines up with the right use case (no cross-wiring):


,use_case_name,domain_industry,objective,embed_model
use_case_key,,,,
carbon_capture,Carbon Capture,,Find amine-based carbon capture approaches tha...,sentence-transformers/paraphrase-multilingual-...
cement_binders,Low-carbon cement binders,Cement & construction materials,Find alternative binder chemistries at TRL 4–8...,sentence-transformers/paraphrase-multilingual-...
ner,Named Entity Recognition,Market intelligence / information extraction,"Improve Precision, Recall, and F1 scores for n...",sentence-transformers/paraphrase-multilingual-...
soil_microbiome,High quality microbial and fungal community in...,Agriculture,Identify research on microbial and fungal comm...,sentence-transformers/paraphrase-multilingual-...
solar_leo,Solar Cells for Low Earth Orbit Satellites,solar cells for low earth orbit satellites,Scout perovskite solar cells and silicon-based...,sentence-transformers/paraphrase-multilingual-...
tech_forecasting,Predicting Technology Developments,Heavy Industries,Explore methods for predicting technological b...,sentence-transformers/paraphrase-multilingual-...
